In [15]:

import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F


In [16]:

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print("using device:", device)


using device: cuda


In [17]:
# Cell 3: attention
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        self.register_buffer(
            "mask",
            torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size),
        )

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)              # (B, T, 3C)
        q, k, v = qkv.split(C, dim=2)  # each: (B, T, C)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))  # (B, nh, T, T)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        out = att @ v                           # (B, nh, T, hd)
        out = out.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        out = self.resid_dropout(self.proj(out))
        return out


In [18]:
# Cell 4: MLP
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [19]:
# Cell 5: transformer block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [20]:
# Cell 6: model
class Max(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            *[TransformerBlock(n_embd, n_heads, block_size, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")

        pos = torch.arange(0, T, device=idx.device)
        x = self.token_emb(idx) + self.pos_emb(pos)[None, :, :]
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss


In [21]:
# Cell 7: load data + tokenizer (char-level)
PROJECT_ROOT = Path("D:/Coding/Chatapp/backend/Max")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "tinystories.txt"

text = DATA_PATH.read_text(encoding="utf-8")
text = text[:5_000_000]  # optional cap for speed

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
default_token_id = stoi.get(" ", 0)

def encode(s):
    return [stoi.get(c, default_token_id) for c in s]

def decode(tokens):
    return "".join(itos[i] for i in tokens)

data = torch.tensor(encode(text), dtype=torch.long)
split_idx = int(0.9 * len(data))
train_data = data[:split_idx]
val_data = data[split_idx:]

print("vocab_size:", vocab_size)
print("train tokens:", len(train_data), "val tokens:", len(val_data))


vocab_size: 73
train tokens: 4500000 val tokens: 500000


In [22]:
# Cell 8: training helpers
# 1) Better hyperparameters (quality-focused)
batch_size = 4               # keep small for VRAM
grad_accum_steps = 8         # effective batch = 32
block_size = 256             # longer context
max_iters = 100000
eval_interval = 300
eval_iters = 100

learning_rate = 3e-4


n_embd = 384
n_heads = 6                  # must divide n_embd
n_layers = 8                 # deeper network
dropout = 0.05


def get_batch(split):
    source = train_data if split == "train" else val_data
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i : i + block_size] for i in ix])
    y = torch.stack([source[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate_metrics(model):
    model.eval()
    out = {}

    for split in ["train", "val"]:
        losses = []
        correct = 0
        total = 0

        for _ in range(eval_iters):
            xb, yb = get_batch(split)
            logits, loss = model(xb, yb)
            losses.append(loss.item())

            preds = logits.argmax(dim=-1)   # (B, T)
            correct += (preds == yb).sum().item()
            total += yb.numel()

        out[split] = {
            "loss": sum(losses) / len(losses),
            "acc_pct": 100.0 * correct / total
        }

    model.train()
    return out



In [9]:
# Cell 9: train
model = Max(
    vocab_size=vocab_size,
    n_embd=n_embd,
    n_heads=n_heads,
    n_layers=n_layers,
    block_size=block_size,
    dropout=dropout,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)
best_val = float("inf")
CKPT_PATH = PROJECT_ROOT / "checkpoints" / "gpt_char.pt"
CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)

for step in range(max_iters + 1):
    if step % eval_interval == 0 or step == max_iters:
        losses = estimate_metrics(model)
        print(
    f"step {step:4d} | "
    f"train loss {losses['train']['loss']:.4f} | train acc {losses['train']['acc_pct']:.2f}% | "
    f"val loss {losses['val']['loss']:.4f} | val acc {losses['val']['acc_pct']:.2f}%"
)
        if losses["val"]["loss"] < best_val:
            best_val = losses["val"]["loss"]
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": {
                        "vocab_size": vocab_size,
                        "block_size": block_size,
                        "n_embd": n_embd,
                        "n_heads": n_heads,
                        "n_layers": n_layers,
                        "dropout": dropout,
                    },
                    "stoi": stoi,
                    "itos": itos,
                },
                CKPT_PATH,
            )
            print("saved:", CKPT_PATH)

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()


step    0 | train loss 4.2827 | train acc 1.76% | val loss 4.2861 | val acc 1.81%
saved: D:\Coding\Chatapp\backend\Max\checkpoints\gpt_char.pt
step  300 | train loss 2.2967 | train acc 31.89% | val loss 2.2891 | val acc 32.36%
saved: D:\Coding\Chatapp\backend\Max\checkpoints\gpt_char.pt
step  600 | train loss 2.1623 | train acc 34.62% | val loss 2.1700 | val acc 34.38%
saved: D:\Coding\Chatapp\backend\Max\checkpoints\gpt_char.pt
step  900 | train loss 1.8252 | train acc 44.29% | val loss 1.8363 | val acc 43.99%
saved: D:\Coding\Chatapp\backend\Max\checkpoints\gpt_char.pt
step 1200 | train loss 1.5843 | train acc 51.32% | val loss 1.5663 | val acc 51.76%
saved: D:\Coding\Chatapp\backend\Max\checkpoints\gpt_char.pt
step 1500 | train loss 1.4346 | train acc 55.78% | val loss 1.4237 | val acc 56.09%
saved: D:\Coding\Chatapp\backend\Max\checkpoints\gpt_char.pt
step 1800 | train loss 1.3433 | train acc 58.61% | val loss 1.3368 | val acc 58.95%
saved: D:\Coding\Chatapp\backend\Max\checkpoints

In [23]:
# Cell 10: generate text
@torch.no_grad()
def generate(prompt="Once upon a time", max_new_tokens=300):
    model.eval()
    idx = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]
        logits, _ = model(idx_cond)
        probs = torch.softmax(logits[:, -1, :], dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return decode(idx[0].tolist())

print(generate("Once upon a time", 500))


NameError: name 'model' is not defined